# Experiment 4 - Native Gold Standard (Bengali)

Fine-tune and evaluate directly on **native Bengali** splits.

| Condition | Backbone | Train | Test |
|---|---|---|---|
| **4a** controlled | `FacebookAI/xlm-roberta-base` | bn-train | bn-test |
| **4b** specialized | `csebuetnlp/banglabert` | bn-train | bn-test |

Both conditions use the **same stratified splits** (seed 42), so the only
difference is the backbone. 4a is the controlled condition comparable to
configs 1-3; 4b is the native-specialized oracle and is reported as
non-controlled.

Each condition is run with 3 seeds (42, 43, 44).
Metrics: macro-F1 (primary), per-class F1, QWK, MAE, and the
upgrade/ok/downgrade error-direction matrix.

`transformers==5.18.0` is already the version Colab's current runtime ships,
so the install cell below is a no-op and nothing preinstalled is disturbed.

**Models are pushed to a private Hugging Face Hub repo after every run**
(~1.1 GB for 4a, ~442 MB for 4b; ~4.7 GB total). Colab's VM is wiped when the
runtime recycles, so this is what keeps the trained weights - which RQ2 (SHAP
attributions) will need - from being lost. The repo is created **private**, since
the source datasets state no licence.

In [ ]:
# Runtime check: this experiment needs a GPU.
import torch, transformers

print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device       :", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU found. In Colab go to Runtime > Change runtime type > "
        "Hardware accelerator = GPU, then re-run this cell."
    )

In [ ]:
# Install pinned deps. torch is deliberately NOT installed so Colab's
# CUDA build stays intact.
!pip install -q "transformers==5.18.0" accelerate openpyxl pandas scikit-learn

In [ ]:
# Get the project. Idempotent: safe to re-run after a runtime restart.
# The datasets are tracked in the repo, so the clone brings code + data.
import os, subprocess, pathlib

if not pathlib.Path("NLP").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/iammatthi/NLP.git"], check=True)
os.chdir("NLP")
print("working dir:", os.getcwd())

data_file = pathlib.Path("data/raw/depression-severity-bengali.xlsx")
assert data_file.exists(), f"{data_file} missing - the clone did not bring the data"
print("dataset OK:", data_file, f"({data_file.stat().st_size:,} bytes)")

In [ ]:
# Persist trained checkpoints to a private Hugging Face Hub repo.
# Set HF_TOKEN in Colab Secrets (key icon in the left sidebar) - it is read
# from there and never printed. If no secret is set, you get a login prompt.
import os

HF_REPO = os.environ.setdefault("HF_REPO", "iammatthi/nlp-exp4-native-bn")
print("HF_REPO =", HF_REPO)

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    print("HF_TOKEN: read from Colab Secrets")
except Exception:
    print("HF_TOKEN: no Colab Secret found, falling back to a login prompt")

from huggingface_hub import login, whoami
if token:
    login(token=token)
else:
    from huggingface_hub import notebook_login
    notebook_login()

print("authenticated as:", whoami()["name"])
print("keep the repo private: the source datasets state no licence")

In [ ]:
# CPU-only build check: verifies both backbones load and produce (n, 4) logits.
# Exits non-zero on failure.
!python scripts/smoke_test_model.py

In [ ]:
# Stratified 80/10/10 splits, fixed seed. Regenerated every session; the seed
# makes them byte-reproducible so runs stay comparable.
!python scripts/prepare_splits.py --lang bn --seed 42

## Train

Class-weighted cross-entropy is on by default (inverse-frequency from the train
split) to counteract the label prior without resampling. Checkpoints are
selected on validation macro-F1; the best one is written to `<output-dir>/final/`
and pushed to the Hub immediately.

Each run takes ~20 min on a T4.

In [ ]:
%%bash
set -e
# Condition 4a - XLM-R on native Bengali (controlled)
python train.py --model xlmr --lang bn --output-dir outputs/4a_xlmr_seed42 --seed 42 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16 && python scripts/push_checkpoint.py --run-dir outputs/4a_xlmr_seed42
python train.py --model xlmr --lang bn --output-dir outputs/4a_xlmr_seed43 --seed 43 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16 && python scripts/push_checkpoint.py --run-dir outputs/4a_xlmr_seed43
python train.py --model xlmr --lang bn --output-dir outputs/4a_xlmr_seed44 --seed 44 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16 && python scripts/push_checkpoint.py --run-dir outputs/4a_xlmr_seed44

In [ ]:
%%bash
set -e
# Condition 4b - BanglaBERT on native Bengali (specialized)
python train.py --model banglabert --lang bn --output-dir outputs/4b_banglabert_seed42 --seed 42 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16 && python scripts/push_checkpoint.py --run-dir outputs/4b_banglabert_seed42
python train.py --model banglabert --lang bn --output-dir outputs/4b_banglabert_seed43 --seed 43 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16 && python scripts/push_checkpoint.py --run-dir outputs/4b_banglabert_seed43
python train.py --model banglabert --lang bn --output-dir outputs/4b_banglabert_seed44 --seed 44 --epochs 4 --train-batch-size 16 --learning-rate 2e-5 --bf16 && python scripts/push_checkpoint.py --run-dir outputs/4b_banglabert_seed44

## Evaluate on the held-out test split

In [ ]:
%%bash
set -e
python evaluate.py --model-dir outputs/4a_xlmr_seed42/final --lang bn --split test --seed 42
python evaluate.py --model-dir outputs/4a_xlmr_seed43/final --lang bn --split test --seed 42
python evaluate.py --model-dir outputs/4a_xlmr_seed44/final --lang bn --split test --seed 42
python evaluate.py --model-dir outputs/4b_banglabert_seed42/final --lang bn --split test --seed 42
python evaluate.py --model-dir outputs/4b_banglabert_seed43/final --lang bn --split test --seed 42
python evaluate.py --model-dir outputs/4b_banglabert_seed44/final --lang bn --split test --seed 42

# metrics are tiny - push them across too
python scripts/push_checkpoint.py --run-dir outputs/4a_xlmr_seed42 --metrics-only
python scripts/push_checkpoint.py --run-dir outputs/4a_xlmr_seed43 --metrics-only
python scripts/push_checkpoint.py --run-dir outputs/4a_xlmr_seed44 --metrics-only
python scripts/push_checkpoint.py --run-dir outputs/4b_banglabert_seed42 --metrics-only
python scripts/push_checkpoint.py --run-dir outputs/4b_banglabert_seed43 --metrics-only
python scripts/push_checkpoint.py --run-dir outputs/4b_banglabert_seed44 --metrics-only

## Summarise across seeds

In [ ]:
import glob, json, statistics as st

def mean_std(vals):
    m = st.mean(vals)
    s = st.stdev(vals) if len(vals) > 1 else 0.0
    return f"{m:.4f} +/- {s:.4f}"

rows = []
for tag in ("4a_xlmr", "4b_banglabert"):
    runs = [json.load(open(p)) for p in sorted(glob.glob(f"outputs/{tag}_seed*/final/eval_test.json"))]
    if not runs:
        print(f"{tag}: no results"); continue
    row = {"condition": tag, "n_seeds": len(runs)}
    for key in ("macro_f1", "qwk", "mae"):
        row[key] = mean_std([r[key] for r in runs])
    row["downgrade_rate"] = mean_std([r["direction"]["downgrade_rate"] for r in runs])
    row["upgrade_rate"] = mean_std([r["direction"]["upgrade_rate"] for r in runs])
    rows.append(row)

for row in rows:
    print(f"{row['condition']:16s} n={row['n_seeds']}")
    for k in ("macro_f1", "qwk", "mae", "downgrade_rate", "upgrade_rate"):
        print(f"  {k:16s} {row[k]}")

# per-class F1 for the paper table
print("\nper-class F1 (mean across seeds):")
for tag in ("4a_xlmr", "4b_banglabert"):
    runs = [json.load(open(p)) for p in sorted(glob.glob(f"outputs/{tag}_seed*/final/eval_test.json"))]
    if not runs: continue
    print(f"  {tag}:")
    for cls in ("minimal", "mild", "moderate", "severe"):
        print(f"    {cls:9s} {mean_std([r[cls] for r in runs])}")

In [ ]:
# Export metrics to the local machine. Checkpoints are large and live on the
# Hub - this zip is just the small numbers.
import glob, zipfile

paths = sorted(
    glob.glob("outputs/*/final/eval_test.json")
    + glob.glob("outputs/*/train_metrics.json")
)
assert paths, "no results found - run the training and evaluation cells first"

with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p)

print(f"zipped {len(paths)} files:")
for p in paths:
    print(" ", p)

from google.colab import files
files.download("results.zip")
print("downloaded results.zip")

In [ ]:
# What ended up on the Hub. Expect 6 run directories (~4.7 GB total).
# Only the best checkpoint per run is kept; intermediate checkpoint-* folders
# are deliberately not uploaded.
from huggingface_hub import HfApi
api = HfApi()
info = api.repo_info(HF_REPO, files_metadata=True)
print(f"repo: {info.id}  (private={info.private})")
total = 0
for s in sorted(info.siblings, key=lambda s: s.rfilename):
    size = getattr(s, "size", 0) or 0
    total += size
    print(f"  {size/1e6:8.1f} MB  {s.rfilename}")
print(f"  total {total/1e6:.1f} MB")

## Notes

- **Where the models live:** a **private** Hugging Face Hub repo, default
  `iammatthi/nlp-exp4-native-bn` (change `HF_REPO` in the auth cell). Each run writes
  `outputs/<tag>_seed<N>/final/` (best checkpoint by validation macro-F1) and
  is pushed as `<run>/final/`, plus `train_metrics.json` and
  `val_predictions.npz`. Intermediate `checkpoint-*` folders stay on the VM only
  and are deliberately not uploaded.
- **Authentication:** set `HF_TOKEN` in Colab Secrets (key icon, left sidebar).
  The token is read from there and never printed. Create one at
  https://huggingface.co/settings/tokens with `write` scope.
- **Keep it private.** The source datasets state no licence, so model weights
  trained on them should not be published until that is settled.
- **Sizes:** 4a (XLM-R) ~1.11 GB per seed, 4b (BanglaBERT) ~442 MB per seed.
  Six runs = ~4.7 GB, which fits in the free 100 GB private quota.
- **Expected runtime:** ~20 min per training run on a T4. Six runs = ~2 hours.
  Because checkpoints are pushed after every run, a crash or idle-timeout costs
  at most one run.
- **Re-running after a runtime restart:** re-run all cells. Cell 4 re-clones,
  cell 5 re-authenticates, and already-pushed runs can be skipped by deleting
  their `!python train.py` lines (or just re-run - results are deterministic
  per seed).
- **Resuming elsewhere:** `python scripts/push_checkpoint.py --run-dir ...`
  and `huggingface_hub.hf_hub_download` work from any machine, so the checkpoints
  are portable off Colab entirely.
- **RQ2:** the SHAP analysis needs these trained weights. Do not delete the repo
  until attribution work is finished.
- **Ablation:** `--no-class-weights` disables the weighted loss.
- Full replication commands, dataset provenance and metrics rationale are in
  `README.md` -> *Replicating experiment 4*.